# 15 - SurrealDB 3.0 Features

## What's new in SurrealDB 3.0

SurrealDB 3.0 introduces several powerful features. This notebook covers the
three major additions supported by the ORM:

1. **Refresh Tokens** — Long-lived tokens for seamless re-authentication via `WITH REFRESH`
2. **DEFINE API** — Server-side REST endpoint definitions
3. **ReferencesField** — Referential integrity with `ON DELETE` strategies

> **Requires:** SurrealDB 3.0+ and `surreal-orm >= 0.30.0`

## Prerequisites

- SurrealDB **3.0+** running locally (see `00_setup.ipynb`)
- Python packages installed (`uv sync` in the project root)

In [ ]:
# -- Setup -----------------------------------------------------------------
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
from dotenv import load_dotenv
load_dotenv()

from src.surreal_orm import SurrealDBConnectionManager

SurrealDBConnectionManager.set_connection(
    os.getenv("SURREALDB_URL", "ws://localhost:8000"),
    os.getenv("SURREALDB_USER", "root"),
    os.getenv("SURREALDB_PASS", "root"),
    os.getenv("SURREALDB_NAMESPACE", "ns"),
    os.getenv("SURREALDB_DATABASE", "db"),
)
print("Connection configured.")

---

## 1. Refresh Tokens

In SurrealDB 2.x, when a JWT access token expired the user had to sign in again
with their credentials. SurrealDB 3.0 adds **refresh tokens** — long-lived tokens
that can be exchanged for new access tokens without re-entering credentials.

### How it works

1. Add `WITH REFRESH` to your `DEFINE ACCESS` statement
2. `signup()` and `signin()` return an `AuthResult` with a `refresh_token` field
3. When the access token expires, call `refresh_access_token()` with the refresh token
4. The server returns a new access token **and** a new refresh token (token rotation)

Token rotation means each refresh token can only be used once — the old one is
automatically revoked by the server.

In [ ]:
from src.surreal_orm import BaseSurrealModel, SurrealConfigDict
from src.surreal_orm.types import TableType
from src.surreal_orm.fields import Encrypted
from src.surreal_orm.auth import AuthenticatedUserMixin


class User(AuthenticatedUserMixin, BaseSurrealModel):
    model_config = SurrealConfigDict(
        table_name="users",
        table_type=TableType.USER,
        access_name="account",
        with_refresh=True,       # Enable refresh tokens (SurrealDB 3.0+)
        grant_duration="30d",    # Refresh token lifetime
    )

    id: str | None = None
    email: str
    password: Encrypted
    name: str


print("User model defined with with_refresh=True")

### Setting up the table and access definition

The ORM provides two methods to set up a USER table without writing raw SQL:

- **`define_table()`** — Generates `DEFINE TABLE` + `DEFINE FIELD` from the model
- **`define_access()`** — Generates `DEFINE ACCESS` with `WITH REFRESH` from the config

The `with_refresh=True` and `grant_duration="30d"` settings in `SurrealConfigDict`
produce the correct `WITH REFRESH` and `DURATION FOR GRANT` clauses automatically.

In [ ]:
# Create table schema from the model definition — no raw SQL needed
table_sql = await User.define_table()
print("Table schema:")
print(table_sql)
print()

# Create the access definition with WITH REFRESH
access_sql = await User.define_access()
print("Access definition:")
print(access_sql)

### AuthResult — Accessing the refresh token

`signup()` and `signin()` now return an `AuthResult` object. It is
backward-compatible with the old `(user, token)` tuple unpacking,
but also carries the `refresh_token` field.

In [ ]:
# Signup — get access token + refresh token
result = await User.signup(
    email="alice@example.com",
    password="secure_password_123",
    name="Alice",
)

print(f"User: {result.user.name} ({result.user.email})")
print(f"Access token: {result.token[:50]}...")
print(f"Refresh token: {result.refresh_token}")
print(f"Refresh token prefix: {result.refresh_token[:16] if result.refresh_token else 'N/A'}")
print()

# Backward-compatible unpacking still works
user, token = result
print(f"Unpacked user: {user.name}")
print(f"Unpacked token: {token[:50]}...")

In [ ]:
# Signin — also returns refresh token
signin_result = await User.signin(
    email="alice@example.com",
    password="secure_password_123",
)

print(f"Signed in as: {signin_result.user.name}")
print(f"Access token: {signin_result.token[:50]}...")
print(f"Refresh token: {signin_result.refresh_token}")

### Refreshing an access token

When the short-lived access token expires (e.g., after 15 minutes), exchange
the refresh token for a new pair of tokens. The old refresh token is revoked
automatically (token rotation).

In [ ]:
# Exchange refresh token for new access + refresh tokens
refresh_result = await User.refresh_access_token(signin_result.refresh_token)

print(f"New access token: {refresh_result.token[:50]}...")
print(f"New refresh token: {refresh_result.refresh_token}")
print(f"User: {refresh_result.user.name}")
print()

# The old refresh token has been revoked — token rotation
print(f"Old refresh token (revoked): {signin_result.refresh_token}")
print(f"New refresh token (active):  {refresh_result.refresh_token}")
print("These are different — the old one can no longer be used.")

In [ ]:
# Clean up auth resources
await User.raw_query("DELETE FROM users")
await User.raw_query("REMOVE ACCESS account ON DATABASE")
await User.raw_query("REMOVE TABLE users")
print("Auth cleanup complete.")

---

## 2. DEFINE API — Server-Side REST Endpoints

SurrealDB 3.0 introduces `DEFINE API`, which lets you define REST API endpoints
directly in the database. These endpoints execute SurrealQL when called via HTTP.

The ORM provides `DefineApi` and `RemoveApi` migration operations for managing
these endpoints in your migration files.

### Syntax

```sql
DEFINE API "/users/list" FOR get
    PERMISSIONS FULL
    THEN { SELECT * FROM users; };
```

The endpoint is then accessible at `http://your-db/api/users/list`.

In [ ]:
from src.surreal_orm.migrations.operations import DefineApi, RemoveApi

# Example: Define a GET endpoint to list all products
list_api = DefineApi(
    name="/products/list",
    method="get",
    handler="SELECT * FROM products ORDER BY name",
)

print("Generated SQL:")
print(list_api.forwards())
print()
print(f"Rollback SQL: {list_api.backwards()}")
print(f"Description: {list_api.describe()}")

In [ ]:
# Example: POST endpoint with permissions
create_api = DefineApi(
    name="/products/create",
    method="post",
    permissions="$auth.role = 'admin'",
    handler="CREATE products SET name = $body.name, price = $body.price",
)

print("Generated SQL:")
print(create_api.forwards())
print()

# Example: API with middleware
protected_api = DefineApi(
    name="/admin/stats",
    method="get",
    middleware=["rate_limiter", "auth_check"],
    handler="SELECT count() FROM users GROUP ALL",
    comment="Admin statistics endpoint",
)

print("With middleware:")
print(protected_api.forwards())

In [ ]:
# Using DefineApi in a migration
# In your migration file (e.g., migrations/0002_add_api_endpoints.py):
#
#   from surreal_orm.migrations.operations import DefineApi, RemoveApi
#
#   operations = [
#       DefineApi(
#           name="/users/list",
#           method="get",
#           handler="SELECT * FROM users WHERE is_active = true",
#       ),
#       DefineApi(
#           name="/users/create",
#           method="post",
#           permissions="$auth.role = 'admin'",
#           handler="CREATE users SET email = $body.email, name = $body.name",
#       ),
#   ]

# RemoveApi for rollback or cleanup
remove = RemoveApi(name="/products/list")
print(f"Remove SQL: {remove.forwards()}")
print(f"Description: {remove.describe()}")

### Applying DEFINE API to the database

You can apply the generated SQL directly using `raw_query()`, or include
the `DefineApi` operations in your migration files for proper versioning.

In [ ]:
# Create a products table and apply the API endpoint
await BaseSurrealModel.raw_query("""
    DEFINE TABLE products SCHEMAFULL;
    DEFINE FIELD name ON products TYPE string;
    DEFINE FIELD price ON products TYPE float;
""")

# Apply the API definition
await BaseSurrealModel.raw_query(list_api.forwards())
print(f"API endpoint created: {list_api.name}")

# Insert sample data
await BaseSurrealModel.raw_query("""
    CREATE products SET name = 'Widget', price = 9.99;
    CREATE products SET name = 'Gadget', price = 24.99;
""")
print("Sample data inserted.")
print()
print("The endpoint is now accessible at: http://localhost:8000/api/products/list")

In [ ]:
# Clean up API resources
await BaseSurrealModel.raw_query(remove.forwards())  # REMOVE API "/products/list"
await BaseSurrealModel.raw_query("REMOVE TABLE products")
print("API cleanup complete.")

---

## 3. ReferencesField — Referential Integrity

SurrealDB 3.0 adds the `REFERENCE` clause to `DEFINE FIELD`. This enables
automatic referential integrity — the database tracks which records reference
which other records, and can enforce `ON DELETE` strategies.

### ON DELETE strategies

| Strategy   | Behavior                                          |
|------------|---------------------------------------------------|
| `IGNORE`   | Default. Referenced records can be deleted freely. |
| `CASCADE`  | Delete the referencing record when the target is deleted. |
| `REJECT`   | Prevent deletion of the target if references exist. |
| `UNSET`    | Set the reference field to `NONE` when the target is deleted. |

### ORM syntax

```python
from surreal_orm.fields import ReferencesField

class Author(BaseSurrealModel):
    books: ReferencesField["books"]                # REFERENCE (default: IGNORE)
    publisher: ReferencesField["publishers", "CASCADE"]  # REFERENCE ON DELETE CASCADE
```

In [ ]:
from src.surreal_orm.fields import ReferencesField
from src.surreal_orm.fields.references import is_references_field, get_references_info, get_references_on_delete


class Author(BaseSurrealModel):
    """An author with tracked book references."""
    model_config = SurrealConfigDict(table_name="authors")

    id: str | None = None
    name: str
    books: ReferencesField["books"]  # Tracks references, ON DELETE IGNORE


class Book(BaseSurrealModel):
    """A book with a reference to its author."""
    model_config = SurrealConfigDict(table_name="books")

    id: str | None = None
    title: str
    author: ReferencesField["authors", "CASCADE"]  # Delete book when author is deleted


print("Models defined: Author, Book")
print()

In [ ]:
# Inspect the field metadata using detection helpers
import typing

# Check Author.books
books_type = typing.get_type_hints(Author, include_extras=True)["books"]
print(f"Author.books:")
print(f"  Is references field: {is_references_field(books_type)}")
print(f"  References table:    {get_references_info(books_type)}")
print(f"  ON DELETE strategy:  {get_references_on_delete(books_type) or 'IGNORE (default)'}")
print()

# Check Book.author
author_type = typing.get_type_hints(Book, include_extras=True)["author"]
print(f"Book.author:")
print(f"  Is references field: {is_references_field(author_type)}")
print(f"  References table:    {get_references_info(author_type)}")
print(f"  ON DELETE strategy:  {get_references_on_delete(author_type)}")

### Generated migration SQL

When you run `surreal-orm makemigrations`, `ReferencesField` generates
the `REFERENCE` clause on the `DEFINE FIELD` statement:

```sql
-- Author.books (no ON DELETE = IGNORE)
DEFINE FIELD books ON authors TYPE option<array<record<books>>> REFERENCE;

-- Book.author (ON DELETE CASCADE)
DEFINE FIELD author ON books TYPE option<record<authors>> REFERENCE ON DELETE CASCADE;
```

This means:
- When a book is deleted, the `books` field on the author is updated automatically
- When an author is deleted, all books referencing that author are also deleted (`CASCADE`)

In [ ]:
# Demonstrate all ON DELETE strategies
print("ON DELETE strategy examples:\n")

strategies = {
    'ReferencesField["orders"]': "IGNORE (default) — referenced records can be deleted freely",
    'ReferencesField["orders", "CASCADE"]': "CASCADE — delete referencing record when target is deleted",
    'ReferencesField["orders", "REJECT"]': "REJECT — prevent deletion if references exist",
    'ReferencesField["orders", "UNSET"]': "UNSET — set field to NONE when target is deleted",
}

for field_def, description in strategies.items():
    print(f"  {field_def}")
    print(f"    → {description}")
    print()

---

## Summary

| Feature | ORM API | SurrealDB Syntax |
|---------|---------|------------------|
| Table setup | `User.define_table()` | `DEFINE TABLE` + `DEFINE FIELD` |
| Access setup | `User.define_access()` | `DEFINE ACCESS ... TYPE RECORD` |
| Refresh Tokens | `with_refresh=True`, `User.refresh_access_token()` | `WITH REFRESH`, `DURATION FOR GRANT` |
| REST Endpoints | `DefineApi(name, method, handler)` | `DEFINE API "/path" FOR method THEN { ... }` |
| References | `ReferencesField["table", "CASCADE"]` | `DEFINE FIELD ... REFERENCE ON DELETE CASCADE` |

### Key points

- **`define_table()`** creates the table schema from the model — no raw SQL needed
- **`define_access()`** creates the auth rules — supports `with_refresh` and custom durations
- **Refresh tokens** use token rotation — always store the latest refresh token
- **`AuthResult`** is backward-compatible: `user, token = await User.signup(...)` still works
- **`DefineApi`** operations belong in migration files for proper versioning
- **`ReferencesField`** is a type annotation — all enforcement is server-side
- All three features require **SurrealDB 3.0+**